## data exploration on resume_df

In [1]:
import pandas as pd

In [ ]:
resume_df = pd.read_csv('datasets/resumes.csv')

In [ ]:
resume_df.size

90

In [ ]:
resume_df.head(10)

,Uniq Id,Crawl Timestamp,Resume Title,Location,Introduction,Work Experience,Education,Skills,Additional Information
0,3ddb29e616f31947053b257f327969d7,2019-02-20 08:47:25 +0000,Sales Manager,"New London, NH",Dynamic technical sales professional with dive...,"Sales Manager-MadgeTech, Inc-August 2015 to Fe...",B.A.-History-Franklin Pierce University-Rindge...,"120 months-CRM,72 months-Contract Negotiation,...",• Well-Developed Sales & Business Acumen ...
1,9138476c76bcbbefadedd4862966c3d2,2019-02-20 07:47:48 +0000,Implementation Engineer,"Worcester, MA","Experienced, dependable and motivated IT Techn...",Implementation Engineer-Versatile Communicatio...,"--ShoreTel University-Austin, TX|Master-PC & N...","15 months-CISCO,12 months-FIBER OPTIC,6 months...","TECHNICAL SKILLS\n\nHardware: Switches, Router..."
2,cd1cafa706f917a627982bf47291b888,2019-02-20 07:37:41 +0000,Education Information Dissemination Coordinator,"Bristol, NH",NaN,Education Information Dissemination Coordinato...,"Bachelor's-Management-Regis College-Weston, MA",NaN,NaN
3,53aea69598c6c1084e4bce89f0494bc3,2019-02-20 08:20:06 +0000,Engineering Department Intern,"Billerica, MA",To obtain full time employment in the field of...,Engineering Department Intern-Town of Billeric...,Bachelor of Science-Civil and Environmental En...,NaN,• Bachelors of Science in Civil and Environmen...
4,90f8f99d66ebc6c09fceee37aff14bc1,2019-02-20 08:35:33 +0000,Pack and Ship/SORT Technician,"Worcester, MA",NaN,Pack and Ship/SORT Technician-Intel Corporatio...,BS-Information Technology-University of Massac...,NaN,Engineering Technician/Planning Analyst/Operat...
5,fb1c760fd5311dad01e950f3062a9296,2019-02-20 06:57:56 +0000,BDC Data Analyst,"Springfield, MA",NaN,BDC Data Analyst-Gary Rome Auto Group-January ...,Bachelor of Science-Psychology-University of P...,"30 months-SIX-SIGMA,36 months-DATA ANALYSIS,24...",CORE COMPETENCIES\n• Project Management\t\t• T...
6,7795cdc183ab88aaffec73a5d99a1b82,2019-02-20 07:27:41 +0000,Safety Engineer Intern,"Boston, MA",NaN,Safety Engineer Intern-Hexagon Manufacturing I...,Master of Science-Systems Engineering-UNIVERSI...,"9 months-MATLAB,36 months-OPTIMIZATION,36 mont...","Core Competencies: Control Systems, Automotive..."
7,e219a144eca8463f5d70fdec9426466f,2019-02-20 08:15:39 +0000,Classified Ads Manager,"Allenstown, NH",To utilize experience and personal skills in t...,Classified Ads Manager-Quality of Life Publica...,NaN,NaN,NaN
8,c7c409fe9652e1eae9ecb05cce05426e,2019-02-20 08:59:32 +0000,ASSISTANT PROGRAM MANAGER,"Pembroke, NH",NaN,ASSISTANT PROGRAM MANAGER-HARBOR HOMES-August ...,CPR certified--SOUTHERN NEW HAMPSHIRE UNIVERISTY-,"13 months-PROGRAM MANAGER,0 months-RETAIL,13 m...",Skills & Abilities\nMANAGEMENT\n• 4 years of m...
9,7f562eaaa992790efe423a197bfd0ffa,2019-02-20 07:31:38 +0000,Technical Customer Service,"Plymouth, MA","High energy, hardworking Engineering graduate ...",Technical Customer Service-SmartCo Services LL...,Master of Science-EILCO-Engineering school of ...,NaN,NaN


In [ ]:
print(resume_df["Additional Information"])

0    •    Well-Developed Sales & Business Acumen   ...
1    TECHNICAL SKILLS\n\nHardware: Switches, Router...
2                                                  NaN
3    • Bachelors of Science in Civil and Environmen...
4    Engineering Technician/Planning Analyst/Operat...
5    CORE COMPETENCIES\n• Project Management\t\t• T...
6    Core Competencies: Control Systems, Automotive...
7                                                  NaN
8    Skills & Abilities\nMANAGEMENT\n• 4 years of m...
9                                                  NaN
Name: Additional Information, dtype: object


In [ ]:
resume_keep_columns = ['Work Experience', 'Skills', 'Education']

In [ ]:
resume_df[resume_keep_columns].head()

,Work Experience,Skills,Education
0,"Sales Manager-MadgeTech, Inc-August 2015 to Fe...","120 months-CRM,72 months-Contract Negotiation,...",B.A.-History-Franklin Pierce University-Rindge...
1,Implementation Engineer-Versatile Communicatio...,"15 months-CISCO,12 months-FIBER OPTIC,6 months...","--ShoreTel University-Austin, TX|Master-PC & N..."
2,Education Information Dissemination Coordinato...,NaN,"Bachelor's-Management-Regis College-Weston, MA"
3,Engineering Department Intern-Town of Billeric...,NaN,Bachelor of Science-Civil and Environmental En...
4,Pack and Ship/SORT Technician-Intel Corporatio...,NaN,BS-Information Technology-University of Massac...


In [ ]:
# create semantic sentence embeddings for job descriptions and resumes
jd_df["semantic_text"] = (
    "Job Title: " + jd_df["job_title"].fillna("") + ". " +
    "Location: " + jd_df["location"].fillna("") + ". " +
    "Description: " + jd_df["job_description"].fillna("") + "."
)

resume_df["semantic_text"] = (
    "Work Experience: " + resume_df["Work Experience"].fillna("") + ". " +
    "Skills: " + resume_df["Skills"].fillna("") + ". " +
    "Education: " + resume_df["Education"].fillna("") + "."
)

In [ ]:
# apply sentence transformer model to create embeddings
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
jd_df["embeddings"] = jd_df["semantic_text"].apply(lambda x: model.encode(x))
resume_df["embeddings"] = resume_df["semantic_text"].apply(lambda x: model.encode(x))

README.md: 0.00B [00:00, ?B/s]

In [ ]:
# embedding dimension = 384
from sentence_transformers import util
import torch

# convert embeddings to torch tensors
jd_embeddings = torch.tensor(np.array(jd_df["embeddings"].tolist()))
resume_embeddings = torch.tensor(np.array(resume_df["embeddings"].tolist()))


cosine_scores = util.cos_sim(resume_embeddings, jd_embeddings)


In [ ]:
top_k = 3
for i in range(len(resume_df)):
    scores = cosine_scores[i]
    top_results = torch.topk(scores, k=top_k)
    print(f"\n=== Resume {i+1} ===")
    for j, score in zip(top_results.indices, top_results.values):
        j = int(j)
        print(f"JD Title: {jd_df.iloc[j]['job_title']}")
        print(f"Location: {jd_df.iloc[j]['location']}")
        print(f"Similarity: {score:.4f}")
        print("-" * 50)


=== Resume 1 ===
JD Title: SolidWorks Product and Portfolio Manager Job in Waltham
Location: Waltham, MA
Similarity: 0.5965
--------------------------------------------------
JD Title: CyberCoders Job Application for Sales Representative - Fiber Optics Solution Selling | Monster.com var MONS_LOG_VARS = {"JobID":
Location: Contact name Lauren Nesmith
Similarity: 0.5948
--------------------------------------------------
JD Title: SolidWorks Product and Portfolio Manager Job in Waltham
Location: Waltham, MA
Similarity: 0.5948
--------------------------------------------------

=== Resume 2 ===
JD Title: Slait Consulting Job Application for Telecommunications Tech | Monster.com var MONS_LOG_VARS = {"JobID":
Location: Contact name Sarah McKernan Phone 0 Fax 0 Address 4505 Cox RdSuite 100Richmond, Virginia 23060
Similarity: 0.6384
--------------------------------------------------
JD Title: Central Office Services Manager - Quantico
Location: Mcb Quantico, VA 22134
Similarity: 0.6289
------